In [2]:
import numpy as np
import pandas as pd

import altair as alt
alt.data_transformers.enable("vegafusion")
import bokeh.io
bokeh.io.output_notebook()
import iqplot

/Users/rf50/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Loading BokehJS ...

**Motivation**

Build a web demo that explains the concepts of differential privacy applied to census data.
Implement simple version of top down algorihtm), and recreate results from this paper: https://www.science.org/doi/pdf/10.1126/sciadv.abk3283.

Do we see a u-shape in population from a post-processing step? compare weighted and unweighted optimization distance loss function to see if u-shape only emerges due to weight protecting 'largest majority group'

- three possibilities that are potentially driving the 'u-shape' that we observe:
    1. clipping heterogeneous blocks
    2. weighted optimization term - where DAS prioritizes accuracy for largest racial group in given are
    3. noising absolutely instead of relatively.

**Methodology notes**

Web demo: toy TopDown algoirthm: 
- true block data -> add DP noise -> post processing -> compare errors
- 2010 Census PL 94-171 redistricting data
- On-Spine:  state -> county -> tract -> block group -> block
- Off-spine: block x WBAHO, VTD can cross blocks

Simulate VTD as random groupings of blocks within a tract 

**Data**  
From block level data, reconstruct with -disolve commands


**Links:**
- IL pop/race data: Redistricting data: df_il: https://redistrictingdatahub.org/state/illinois/
- IL shapefile:
- Kenny paper: https://www.science.org/doi/pdf/10.1126/sciadv.abk3283

# Clean data

In [3]:
df_il_full = pd.read_csv('data/il_pl2010_b/il_pl2010_b.csv')

columns = [
    'STATE',
    'COUNTY',
    'TRACT',
    'BLOCK',
    'POP100', # total population (all voting ages)
    'P0010001', # population from race table
    'P0010003', # → White alone
    'P0010004', # → Black or African American alone
    'P0010005', # → American Indian / Alaska Native
    'P0010006', # → Asian
    'P0010007', # → Native Hawaiian / Pacific Islander
    'P0010008', # → Some other race
    'P0010009', # → Two or more races
    ]

df_il_full = df_il_full[columns]

df_il = df_il_full.copy()
df_il['white'] = df_il_full['P0010003']
df_il['black'] = df_il_full['P0010004']
df_il['asian'] = df_il_full['P0010006']
df_il['other'] = df_il_full['P0010005'] + df_il['P0010007'] + df_il['P0010008'] + df_il['P0010009']
df_il['population_block'] = df_il['P0010001']

df_il = df_il[[
    'STATE', 'COUNTY', 'TRACT', 'BLOCK', 
    'population_block', 'white', 'black', 'asian', 'other']]

/var/folders/6y/m69ngqb568nbb5t5p3lz216r0000gn/T/ipykernel_47327/707147066.py:1: DtypeWarning: Columns (40,41,42) have mixed types. Specify dtype option on import or set low_memory=False.
  df_il_full = pd.read_csv('data/il_pl2010_b/il_pl2010_b.csv')


In [4]:
df_il_block = df_il

df_il_tract = df_il_block.groupby(['STATE','COUNTY','TRACT'])[[
    'population_block',
    'white',
    'black',
    'asian',
    'other'
]].sum().reset_index().rename(columns={'population_block':'population_tract'})

df_il_county = df_il_tract.groupby(['STATE','COUNTY'])[[
    'population_tract',
    'white',
    'black',
    'asian',
    'other'
]].sum().reset_index().rename(columns={'population_tract':'population_county'})


df_il_state = df_il_county.groupby(['STATE'])[[
    'population_county',
    'white',
    'black',
    'asian',
    'other'
]].sum().reset_index().rename(columns={'population_county':'population_state'})

In [5]:
# df_il_state
# df_il_county.head(2)
# df_il_tract.head(2)
df_il_block.head(2)

,STATE,COUNTY,TRACT,BLOCK,population_block,white,black,asian,other
0,17,1,10200,4009,28,25,0,1,2
1,17,1,10200,4010,8,8,0,0,0


In [6]:
block_df = df_il_block.copy()
tract_df = df_il_tract.copy()
county_df = df_il_county.copy()
state_df = df_il_state.copy()

block_df['geoid'] = df_il_block['BLOCK']
block_df['parent_geoid'] = df_il_block['TRACT']
block_df['true_pop'] = df_il_block['population_block']

tract_df['geoid'] = df_il_tract['TRACT']
tract_df['parent_geoid'] = df_il_tract['COUNTY']
tract_df['true_pop'] = df_il_tract['population_tract']

county_df['geoid'] = df_il_county['COUNTY']
county_df['parent_geoid'] = df_il_county['STATE']
county_df['true_pop'] = df_il_county['population_county']

state_df['geoid'] = df_il_state['STATE']
state_df['true_pop'] = df_il_state['population_state']

block_df = block_df[[
    'geoid', 'parent_geoid', 'true_pop', 
    'white','black', 'asian', 'other']]
tract_df = tract_df[[
    'geoid', 'parent_geoid', 'true_pop', 
    'white','black', 'asian', 'other']]
county_df = county_df[[
    'geoid', 'parent_geoid', 'true_pop', 
    'white','black', 'asian', 'other']]
state_df = state_df[[
    'geoid', 'true_pop', 
    'white','black', 'asian', 'other']]

In [7]:
# block_df.head(1)
# tract_df.head(1)
# county_df.head(1)
state_df.head(1)

,geoid,true_pop,white,black,asian,other
0,17,12830632,9177877,1866414,586934,1199407


In [111]:
block_df.to_csv("data/processed_data/DF_IL_2010_BLOCK.csv", index=False)
tract_df.to_csv("data/processed_data/DF_IL_2010_TRACT.csv", index=False)
county_df.to_csv("data/processed_data/DF_IL_2010_COUNTY.csv", index=False)
state_df.to_csv("data/processed_data/DF_IL_2010_STATE.csv", index=False)

In [107]:
# p = iqplot.ecdf(df_il_block, q='population_block')
# bokeh.io.show(p)